# AgriDiagnose Model V2 — isolated TensorFlow 2.15 on Kaggle

Kaggle's notebook kernel may remain Python 3.12 / TensorFlow 2.20. This notebook creates a separate Python 3.11 environment under `/kaggle/working` and launches all Experiment A ML work through that interpreter. It never replaces system packages. Training is disabled by default.

In [ ]:
# 1. Audit the Kaggle system kernel. This is NOT the Experiment A environment.
import json, os, platform, re, shutil, subprocess, sys
from pathlib import Path
import tensorflow as system_tf
import keras as system_keras
import numpy as system_numpy

print('System Python:', sys.version)
print('System OS:', platform.platform())
print('System TensorFlow:', system_tf.__version__)
print('System Keras:', system_keras.__version__)
print('System NumPy:', system_numpy.__version__)
print('System CUDA build:', system_tf.test.is_built_with_cuda())
SYSTEM_GPUS = system_tf.config.list_physical_devices('GPU')
print('System GPUs:', SYSTEM_GPUS)
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)
if not SYSTEM_GPUS:
    raise RuntimeError('KAGGLE_GPU_NOT_AVAILABLE: Settings -> Accelerator -> GPU')

## 2. Clone the approved source revision
Internet must be enabled. The public repository is checked out at one immutable revision; no GitHub token is needed.

In [ ]:
REPOSITORY_URL = 'https://github.com/ihebjdey2/ai-plant-disease-detection.git'
APPROVED_CODE_REVISION = '183bdc0baae18cc02f9d02188462930bff423360'
PROJECT_ROOT = Path('/kaggle/working/ai-plant-disease-detection')
if not re.fullmatch(r'[0-9a-f]{40}', APPROVED_CODE_REVISION):
    raise RuntimeError('APPROVED_CODE_REVISION_NOT_PINNED')
if not (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
subprocess.run(['git', 'fetch', 'origin'], cwd=PROJECT_ROOT, check=True)
subprocess.run(['git', 'checkout', '--detach', APPROVED_CODE_REVISION], cwd=PROJECT_ROOT, check=True)
HEAD = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, check=True, capture_output=True, text=True).stdout.strip()
assert HEAD == APPROVED_CODE_REVISION
print('Approved source revision:', HEAD)

## 3. Bootstrap isolated Python 3.11
The bootstrap uses the official versioned standalone installer for `uv==0.12.3`; it does not need Kaggle host `pip`, `venv`, `ensurepip`, or `pipx`. It downloads uv-managed Python 3.11, creates `agridiagnose-tf215`, and installs the pinned ML requirements below `/kaggle/working`. Python-specific inherited variables are removed for child interpreters while CUDA variables are preserved.

In [ ]:
RUNTIME_ROOT = Path('/kaggle/working/agridiagnose-tf215-runtime')
TF215_PYTHON = RUNTIME_ROOT / 'venvs/agridiagnose-tf215/bin/python'
ISOLATED_ENV = os.environ.copy()
for name in ('PYTHONPATH', 'PYTHONHOME', 'VIRTUAL_ENV'):
    ISOLATED_ENV.pop(name, None)
ISOLATED_ENV['PYTHONNOUSERSITE'] = '1'
subprocess.run([
    sys.executable, '-I', str(PROJECT_ROOT / 'scripts/bootstrap_kaggle_tf215_runtime.py'),
    '--working-root', str(RUNTIME_ROOT), '--project-root', str(PROJECT_ROOT),
], check=True, env=ISOLATED_ENV)
assert TF215_PYTHON.is_file()
print('Isolated interpreter:', TF215_PYTHON)
print('System interpreter remains:', sys.executable)

## 4. Hard TensorFlow 2.15 GPU gate and smoke test
This subprocess must report Python 3.11.x, TensorFlow/Keras 2.15.x, NumPy 1.26.4, a CUDA build, at least one GPU, and matrix multiplication on that GPU. Any failure stops with `KAGGLE_TF215_GPU_RUNTIME_FAILED`.

In [ ]:
RUNTIME_REPORT = RUNTIME_ROOT / 'tf215-gpu-runtime.json'
subprocess.run([
    str(TF215_PYTHON), str(PROJECT_ROOT / 'scripts/run_kaggle_model_v2_experiment_a.py'),
    'verify-runtime', '--output', str(RUNTIME_REPORT),
], check=True, env=ISOLATED_ENV)
ISOLATED_RUNTIME = json.loads(RUNTIME_REPORT.read_text(encoding='utf-8'))
print(json.dumps(ISOLATED_RUNTIME, indent=2))
assert ISOLATED_RUNTIME['status'] == 'TF215_GPU_RUNTIME_VALIDATED'
assert ISOLATED_RUNTIME['training_performed'] is False

## 5. Attach and configure the five private datasets
Use **Add Input → Your Datasets**. Do not attach INTERNAL TEST images or PlantDoc TEST. Edit only the five paths below if Kaggle assigned different slugs.

In [ ]:
print('Available Kaggle inputs:')
for item in sorted(Path('/kaggle/input').iterdir()):
    print(' -', item)

SOURCE_ROOTS = {
    'historical': '/kaggle/input/agridiagnose-historical',
    'pldd_up': '/kaggle/input/agridiagnose-pldd-up',
    'seasonal_corn': '/kaggle/input/agridiagnose-seasonal-corn',
    'plantdoc_train': '/kaggle/input/agridiagnose-plantdoc-train',
    'banu_deb': '/kaggle/input/agridiagnose-banu-deb',
}
START_TRAINING = False
EXECUTION_CONFIG = RUNTIME_ROOT / 'experiment-a-config.json'
CONFIG = {
    'experiment': 'agri-diagnose-v2-exp-a', 'source_roots': SOURCE_ROOTS,
    'batch_size': 32, 'start_training': START_TRAINING,
    'restart_interrupted_phase': False,
    'internal_test_loaded': False, 'plantdoc_test_loaded': False,
}
EXECUTION_CONFIG.write_text(json.dumps(CONFIG, indent=2) + '\n', encoding='utf-8')
print(EXECUTION_CONFIG.read_text(encoding='utf-8'))

## 6. Exhaustive TRAIN / VALIDATION preflight and model audit
The isolated Python 3.11 process verifies 58,857 TRAIN and 7,362 VALIDATION images, all 39 classes, MacroF1, preprocessing, the locked TEST hash, fresh ImageNet MobileNetV2, output 39, and the frozen Phase 1 backbone. It never calls `model.fit()`.

In [ ]:
PREFLIGHT_REPORT = RUNTIME_ROOT / 'experiment-a-preflight.json'
subprocess.run([
    str(TF215_PYTHON), str(PROJECT_ROOT / 'scripts/run_kaggle_model_v2_experiment_a.py'),
    'preflight', '--config', str(EXECUTION_CONFIG), '--output', str(PREFLIGHT_REPORT),
], check=True, env=ISOLATED_ENV)
PREFLIGHT = json.loads(PREFLIGHT_REPORT.read_text(encoding='utf-8'))
DATA = PREFLIGHT['preflight']
assert DATA['train']['expected'] == DATA['train']['resolved'] == 58857
assert DATA['train']['missing'] == DATA['train']['unreadable'] == 0
assert DATA['validation']['expected'] == DATA['validation']['resolved'] == 7362
assert DATA['validation']['missing'] == DATA['validation']['unreadable'] == 0
assert DATA['train_class_coverage'] == DATA['validation_class_coverage'] == 39
assert DATA['internal_test_loaded'] is False and DATA['plantdoc_test_loaded'] is False
assert PREFLIGHT['phase1_model_audit']['backbone_trainable'] is False
assert PREFLIGHT['phase1_model_audit']['output_shape'] == [None, 39]
assert PREFLIGHT['training_performed'] is False
print(json.dumps(PREFLIGHT, indent=2))

## 7. Stop here for runtime compatibility review
The next cell remains blocked. Run it only after a separate human approval to begin Experiment A training. Training still executes in the isolated Python 3.11 subprocess, never in the notebook kernel.

In [ ]:
START_TRAINING = False
if not START_TRAINING:
    print('SAFE STOP: runtime/preflight complete; neural-network training is disabled.')
else:
    CONFIG['start_training'] = True
    EXECUTION_CONFIG.write_text(json.dumps(CONFIG, indent=2) + '\n', encoding='utf-8')
    subprocess.run([
        str(TF215_PYTHON), str(PROJECT_ROOT / 'scripts/run_kaggle_model_v2_experiment_a.py'),
        'train', '--config', str(EXECUTION_CONFIG), '--authorize-training',
    ], check=True, env=ISOLATED_ENV)

## Required outcome now
Keep `START_TRAINING = False`. Preserve `tf215-gpu-runtime.json` and `experiment-a-preflight.json`, then send both reports for review. No candidate model should exist yet.